In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

In [2]:
# load data
df_script = pd.read_csv('df_script.csv')
df_rating = pd.read_csv('df_rating.csv')

In [3]:
# create empty intermediary dataframe to record sentiment, emotion and humor scores for each dialogue line
df_dialogue_emotion = pd.DataFrame(columns=['dialogue_line'])
df_dialogue_sentiment = pd.DataFrame(columns=['dialogue_line'])
df_dialogue_humor = pd.DataFrame(columns=['dialogue_line'])

In [4]:
df_script['dialogue_raw'][4]

'Wait, does he eat chalk?'

## Feature Engineering
We will create the NLP features and attributes, and later look into feature selection

### Restructure the script

In [5]:
# Replace empty dialogue_clean values with dialogue_raw when type is 'dialogue'
df_script.loc[(df_script['type'] == 'dialogue') & (df_script['dialogue_clean'].isna()), 'dialogue_clean'] = df_script.loc[(df_script['type'] == 'dialogue') & (df_script['dialogue_clean'].isna()), 'dialogue_raw']

In [6]:
# Structure script dataset
# Create episode_script column by combining dialogue with speaker and other content
def create_episode_script(group, clean_dialogue):
    script_lines = []

    if clean_dialogue == True:

        for _, row in group.iterrows():

            if row['type'] == 'dialogue' and pd.notna(row['speaker']) and pd.notna(row['dialogue_clean']):
                script_lines.append(f"{row['speaker']}: {row['dialogue_clean']}")
            elif row['type'] == 'scene_note' and pd.notna(row['scene_note_only']):
                script_lines.append(f"[SCENE NOTE: {row['scene_note_only']}]")
            elif row['type'] == 'stage_direction' and pd.notna(row['stage_direction_only']):
                script_lines.append(f"[STAGE DIRECTION: {row['stage_direction_only']}]")    

    else:
    
        for _, row in group.iterrows():
            if row['type'] == 'dialogue' and pd.notna(row['speaker']) and pd.notna(row['dialogue_raw']):
                script_lines.append(f"{row['speaker']}: {row['dialogue_raw']}")
            elif row['type'] == 'scene_note' and pd.notna(row['scene_note_only']):
                script_lines.append(f"[SCENE NOTE: {row['scene_note_only']}]")
            elif row['type'] == 'stage_direction' and pd.notna(row['stage_direction_only']):
                script_lines.append(f"[STAGE DIRECTION: {row['stage_direction_only']}]")
    
    return '\n\n'.join(script_lines)

# Group by season and episode and create script (dialogue_raw)
df_structured = df_script.groupby(['season', 'episode']).apply(create_episode_script, clean_dialogue=False).reset_index()
df_structured.columns = ['season', 'episode', 'episode_script']

# create script (dialogue_clean)
df_structured_clean = df_script.groupby(['season', 'episode']).apply(create_episode_script, clean_dialogue=True).reset_index()
df_structured_clean.columns = ['season', 'episode', 'episode_script_clean']

# Merge with episode metadata
episode_metadata = df_script.groupby(['season', 'episode']).agg({
    'title': 'first'
}).reset_index()

df_structured = df_structured.merge(episode_metadata, on=['season', 'episode'])
# merge clean script to df_structured
df_structured = df_structured.merge(df_structured_clean, on=['season', 'episode'])

# Reorder columns
df_structured = df_structured[['season', 'episode', 'title', 'episode_script', 'episode_script_clean']]

C:\Users\dimit\AppData\Local\Temp\ipykernel_4712\3078336972.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_structured = df_script.groupby(['season', 'episode']).apply(create_episode_script, clean_dialogue=False).reset_index()
C:\Users\dimit\AppData\Local\Temp\ipykernel_4712\3078336972.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_structured_clean = df_script.groupby(['season', 'episode']).apply(cre

In [7]:
df_structured

,season,episode,title,episode_script,episode_script_clean
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo..."
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th..."
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but..."
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ..."
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ..."
...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M..."
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...


### 1. Lexical & Stylistic features

- avg_words_per_line – average dialogue length

- vocab_richness – type-token ratio (unique_words / total_words)

- character_lines_{NAME} – % of lines spoken by each main character (Ross, Monica, Chandler, Joey, Rachel, Phoebe) and also % of lines spoken by characters that are not the main cast

#### 1.1. avg_words_per_line

In [8]:
# calculate average words per line per episode
def avg_words_per_line(script):
    total_words = 0
    total_lines = 0
    lines = script.split('\n\n')
    for line in lines:
        if ': ' in line and not line.startswith('['):  # check if line contains dialogue and doesn't start with [
            dialogue = line.split(': ', 1)[1]  # get the part after the first ': '
            words = dialogue.split()  # split dialogue into words
            total_words += len(words)  # count words and add to total
            total_lines += 1  # count this as a dialogue line
    return total_words / total_lines if total_lines > 0 else 0

# Apply average words per line function to each episode script
df_structured['avg_words_per_line'] = df_structured['episode_script_clean'].apply(avg_words_per_line)

In [9]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908
...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667


#### 1.2. vocab_richness

In [10]:
# calculate vocab richness (unique words / total words) per episode
def vocab_richness(script):
    words = []
    lines = script.split('\n\n')
    for line in lines:
        if ': ' in line and not line.startswith('['):  # check if line contains dialogue and doesn't start with [
            dialogue = line.split(': ', 1)[1]  # get the part after the first ': '
            words.extend(dialogue.split())  # split dialogue into words and add to list
    total_words = len(words)
    unique_words = len(set(words))
    if total_words == 0:
        return 0
    return unique_words / total_words  # calculate vocabulary richness

# Apply vocab richness function to each episode script
df_structured['vocab_richness'] = df_structured['episode_script_clean'].apply(vocab_richness)


In [11]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799
...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879


#### 1.3. character_lines_{NAME}

In [12]:
all_speakers = df_script['speaker'].dropna().unique()

# from all_speakers, extract the ones related to only the main characters
monica_list = [speaker for speaker in all_speakers if 'MONICA' in speaker]
rachel_list = [speaker for speaker in all_speakers if 'RACHEL' in speaker]
ross_list = [speaker for speaker in all_speakers if 'ROSS' in speaker]
chandler_list = [speaker for speaker in all_speakers if 'CHANDLER' in speaker]
joey_list = [speaker for speaker in all_speakers if 'JOEY' in speaker]
phoebe_list = [speaker for speaker in all_speakers if 'PHOEBE' in speaker]

monica_list.remove("MONICA'S BOYFRIEND")

rachel_list.remove("RACHEL’S BOSS")

joey_list.remove("JOEY'S CO-STAR")
joey_list.remove("JOEY'S DATE")
joey_list.remove("JOEY'S HAND TWIN")
joey_list.remove("JOEY'S DOCTOR")
joey_list.remove("JOEY’S SISTER")
joey_list.remove("JOEY’S SISTERS")

phoebe_list.remove("PHOEBE'S FRIENDS")
phoebe_list.remove("PHOEBE'S ASSISTANT")

# combine all main character lists into one list (it should not be list of lists)
lists_by_main = {
    "MONICA": monica_list,
    "RACHEL": rachel_list,
    "ROSS": ross_list,
    "CHANDLER": chandler_list,
    "JOEY": joey_list,
    "PHOEBE": phoebe_list
}

In [13]:
# calculate character_lines_{NAME} columns which calculate
# % of lines spoken by each main character (Ross, Monica, Chandler, Joey, Rachel, Phoebe)
# use the created lists above. if speaker not in any of the lists, count as "Other"
# use df_structured.namely episode_script_clean. if line starts has ": " and does not start with "[", it is a dialogue line
def calculate_character_lines(group, char_list):
    total_lines = 0
    char_lines = 0
    lines = group.split('\n\n')
    for line in lines:
        if ': ' in line and not line.startswith('['):  # check if line contains dialogue and doesn't start with [
            total_lines += 1  # count this as a dialogue line
            speaker = line.split(': ', 1)[0]  # get the part before the first ': '
            if speaker in char_list:
                char_lines += 1  # count this as a line spoken by the character
    if total_lines > 0:
        return char_lines / total_lines
    return 0.0

# Apply the function for each main character
df_structured['character_lines_MONICA'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=monica_list)
df_structured['character_lines_RACHEL'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=rachel_list)
df_structured['character_lines_ROSS'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=ross_list)
df_structured['character_lines_CHANDLER'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=chandler_list)
df_structured['character_lines_JOEY'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=joey_list)
df_structured['character_lines_PHOEBE'] = df_structured['episode_script_clean'].apply(calculate_character_lines, char_list=phoebe_list)

# calculate character_lines_OTHER
def calculate_other_character_lines(group, all_char_lists):
    total_lines = 0
    other_lines = 0
    lines = group.split('\n\n')
    for line in lines:
        if ': ' in line and not line.startswith('['):  # check if line contains dialogue and doesn't start with [
            total_lines += 1  # count this as a dialogue line
            speaker = line.split(': ', 1)[0]  # get the part before the first ': '
            if not any(speaker in char_list for char_list in all_char_lists):
                other_lines += 1  # count this as a line spoken by "Other"
    if total_lines > 0:
        return other_lines / total_lines
    return 0.0

df_structured['character_lines_OTHER'] = df_structured['episode_script_clean'].apply(calculate_other_character_lines, all_char_lists=lists_by_main.values())

In [14]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,character_lines_CHANDLER,character_lines_JOEY,character_lines_PHOEBE,character_lines_OTHER
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,0.133333,0.133333,0.066667,0.110000
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,0.069959,0.037037,0.057613,0.308642
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,0.139535,0.108527,0.147287,0.174419
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,0.130952,0.099206,0.126984,0.130952
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,0.147059,0.151261,0.096639,0.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070,0.106583,0.153605,0.122257,0.094044,0.122257,0.159875,0.244514
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141,0.092437,0.138655,0.214286,0.142857,0.121849,0.134454,0.159664
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001,0.206107,0.167939,0.175573,0.160305,0.114504,0.118321,0.061069
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879,0.146667,0.110000,0.183333,0.156667,0.136667,0.143333,0.130000


### 2. Sentiment & Emotion

- sentiment_mean – average sentiment polarity score (e.g., from VADER or TextBlob)

- emotion_joy, emotion_sadness, emotion_anger, emotion_surprise, emotion_disgust, emotion_fear – % of words/lines carrying each emotion (from NRC Emotion Lexicon)

- sentiment_trajectory_slope – slope of sentiment progression across the episode (start-to-end change)

#### 2.1. emotion_joy, emotion_sadness, emotion_anger, emotion_surprise, emotion_disgust, emotion_fear, emotion_neutral

In [15]:
# Use a pipeline as a high-level helper
from transformers import pipeline

model_emotion = pipeline("text-classification", model="michellejieli/emotion_text_classifier", return_all_scores=True)

model_emotion("I am so happy today!")

d:\DSAI\Y1Q1\Research_Topics_DM\Group_Project\EMM-Friends-Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu
d:\DSAI\Y1Q1\Research_Topics_DM\Group_Project\EMM-Friends-Project\.venv\lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': 'anger', 'score': 0.0008601052104495466},
  {'label': 'disgust', 'score': 0.0001605393917998299},
  {'label': 'fear', 'score': 0.00022762933804187924},
  {'label': 'joy', 'score': 0.9924352765083313},
  {'label': 'neutral', 'score': 0.003412239719182253},
  {'label': 'sadness', 'score': 0.0005046047153882682},
  {'label': 'surprise', 'score': 0.0023996420204639435}]]

In [16]:
from tqdm import tqdm
tqdm.pandas()

def calculate_emotions(script, model):

    global df_dialogue_emotion

    
    emotion_totals = {
        "joy": 0,
        "sadness": 0,
        "anger": 0,
        "fear": 0,
        "surprise": 0,
        "disgust": 0,
        "neutral": 0
    }
    lines = script.split('\n\n')
    dialogue_lines = [line for line in lines if ': ' in line and not line.startswith('[')]  # filter to only dialogue lines
    total_lines = 0
    for line in dialogue_lines:
        total_lines += 1
        dialogue = line.split(': ', 1)[1]  # get the part after the first ': '
        results = model(dialogue)

        # add dialogue line and its emotion scores to df_dialogue_emotion
        df_dialogue_emotion = pd.concat([df_dialogue_emotion, pd.DataFrame([{'dialogue_line': dialogue, **{result['label'].lower(): result['score'] for result in results[0]}}])], ignore_index=True)

        # results is a list of lists of dicts, we take the first element
        for result in results[0]:
            emotion = result['label'].lower()
            score = result['score']
            if emotion in emotion_totals:
                emotion_totals[emotion] += score
        

    # Normalize by total number of dialogue lines
    for key in emotion_totals:
        emotion_totals[key] /= total_lines

    return emotion_totals     

# create columns emotion_{EMOTION} for each emotion
def add_emotion_columns(row, model):
    emotions = calculate_emotions(row['episode_script'], model)
    for emotion, value in emotions.items():
        row[f'emotion_{emotion}'] = value
    return row

df_structured = df_structured.progress_apply(add_emotion_columns, axis=1, model=model_emotion)

100%|██████████| 236/236 [29:11<00:00,  7.42s/it]


#### 2.2. sentiment_mean - neg, neu, pos, compound

In [17]:
# use vader to get sentiment score per dialogue line and then average per episode
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
def calculate_sentiment(script, analyzer):

    global df_dialogue_sentiment

    sentiment_totals = {
        "neg": 0,
        "neu": 0,
        "pos": 0,
        "compound": 0
    }
    lines = script.split('\n\n')
    dialogue_lines = [line for line in lines if ': ' in line and not line.startswith('[')]  # filter to only dialogue lines
    total_lines = 0
    for line in dialogue_lines:
        total_lines += 1
        dialogue = line.split(': ', 1)[1]  # get the part after the first ': '
        scores = analyzer.polarity_scores(dialogue)

        df_dialogue_sentiment = pd.concat([df_dialogue_sentiment, pd.DataFrame([{'dialogue_line': dialogue, **scores}])], ignore_index=True)

        for key in sentiment_totals:
            sentiment_totals[key] += scores[key]
    
    # Normalize by total number of dialogue lines
    for key in sentiment_totals:
        sentiment_totals[key] /= total_lines

    return sentiment_totals

# create columns sentiment_{SENTIMENT} for each sentiment
def add_sentiment_columns(row, analyzer):
    sentiments = calculate_sentiment(row['episode_script'], analyzer)
    for sentiment, value in sentiments.items():
        row[f'sentiment_{sentiment}'] = value
    return row

df_structured = df_structured.apply(add_sentiment_columns, axis=1, analyzer=analyzer)


In [18]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,...,emotion_sadness,emotion_anger,emotion_fear,emotion_surprise,emotion_disgust,emotion_neutral,sentiment_neg,sentiment_neu,sentiment_pos,sentiment_compound
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,...,0.057866,0.074029,0.046573,0.151716,0.058355,0.480126,0.069320,0.773893,0.156783,0.103839
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,...,0.055387,0.044326,0.026094,0.170459,0.056568,0.533810,0.055897,0.786313,0.157794,0.121136
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,...,0.057678,0.061652,0.031391,0.125838,0.070791,0.501233,0.041155,0.775074,0.183779,0.179116
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,...,0.030105,0.052823,0.025490,0.189200,0.046026,0.503466,0.056079,0.785571,0.158349,0.111874
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,...,0.058386,0.037968,0.019409,0.176742,0.074673,0.510663,0.043924,0.796639,0.159445,0.145747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070,0.106583,0.153605,0.122257,...,0.059324,0.057531,0.021107,0.173413,0.042203,0.460592,0.070734,0.752928,0.176342,0.160163
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141,0.092437,0.138655,0.214286,...,0.052343,0.040942,0.043780,0.158035,0.033455,0.496071,0.053908,0.773298,0.172803,0.169121
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001,0.206107,0.167939,0.175573,...,0.070081,0.059062,0.017390,0.162394,0.049853,0.489490,0.046672,0.798347,0.154977,0.142221
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879,0.146667,0.110000,0.183333,...,0.051398,0.052262,0.019799,0.192423,0.034782,0.459105,0.035767,0.743953,0.220270,0.185630


#### 2.3. sentiment_trajectory_slope

In [19]:
# calculate sentiment_trajectory_slope – slope of sentiment progression across the episode (start-to-end change)

def calculate_sentiment_trajectory_slope(script, analyzer):
    sentiment_scores = []
    lines = script.split('\n\n')
    dialogue_lines = [line for line in lines if ': ' in line and not line.startswith('[')]
    
    for line in dialogue_lines:
        dialogue = line.split(': ', 1)[1]
        compound_score = analyzer.polarity_scores(dialogue)["compound"]
        sentiment_scores.append(compound_score)
    
    if len(sentiment_scores) < 2:
        return 0.0
    
    # 1. Smoothed trajectory (moving average to reduce noise)
    window_size = min(10, len(sentiment_scores) // 5)  # adaptive window size
    if window_size > 1:
        smoothed_scores = []
        for i in range(len(sentiment_scores)):
            start_idx = max(0, i - window_size // 2)
            end_idx = min(len(sentiment_scores), i + window_size // 2 + 1)
            smoothed_scores.append(np.mean(sentiment_scores[start_idx:end_idx]))
    else:
        smoothed_scores = sentiment_scores
    
    # 2. Linear regression on smoothed data with normalized positions
    x = np.linspace(0, 1, len(smoothed_scores))  # normalized positions (0 to 1)
    y = np.array(smoothed_scores)
    slope = np.polyfit(x, y, 1)[0]
    
    return slope

# Apply sentiment trajectory slope function to each episode script
df_structured['sentiment_trajectory_slope'] = df_structured['episode_script'].apply(calculate_sentiment_trajectory_slope, analyzer=analyzer)

In [20]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,...,emotion_anger,emotion_fear,emotion_surprise,emotion_disgust,emotion_neutral,sentiment_neg,sentiment_neu,sentiment_pos,sentiment_compound,sentiment_trajectory_slope
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,...,0.074029,0.046573,0.151716,0.058355,0.480126,0.069320,0.773893,0.156783,0.103839,-0.105783
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,...,0.044326,0.026094,0.170459,0.056568,0.533810,0.055897,0.786313,0.157794,0.121136,-0.058521
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,...,0.061652,0.031391,0.125838,0.070791,0.501233,0.041155,0.775074,0.183779,0.179116,-0.076444
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,...,0.052823,0.025490,0.189200,0.046026,0.503466,0.056079,0.785571,0.158349,0.111874,0.059526
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,...,0.037968,0.019409,0.176742,0.074673,0.510663,0.043924,0.796639,0.159445,0.145747,0.088462
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070,0.106583,0.153605,0.122257,...,0.057531,0.021107,0.173413,0.042203,0.460592,0.070734,0.752928,0.176342,0.160163,0.039302
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141,0.092437,0.138655,0.214286,...,0.040942,0.043780,0.158035,0.033455,0.496071,0.053908,0.773298,0.172803,0.169121,0.112704
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001,0.206107,0.167939,0.175573,...,0.059062,0.017390,0.162394,0.049853,0.489490,0.046672,0.798347,0.154977,0.142221,0.020706
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879,0.146667,0.110000,0.183333,...,0.052262,0.019799,0.192423,0.034782,0.459105,0.035767,0.743953,0.220270,0.185630,0.095480


### 3. Humor-related

- humor - % of dialogue lines categorized as funny per episode

#### 3.1. humor

In [21]:
# Use a pipeline as a high-level helper
from transformers import pipeline

model_humor = pipeline("text-classification", model="mohameddhiab/humor-no-humor", return_all_scores=True)

result = model_humor("Wait, does he eat chalk?")
print(result)
predicted_label = max(result[0], key=lambda x: x['score'])['label']
print(f"Predicted label: {predicted_label}")

Device set to use cpu


[[{'label': 'NO_HUMOR', 'score': 0.2743643820285797}, {'label': 'HUMOR', 'score': 0.7256356477737427}]]
Predicted label: HUMOR


d:\DSAI\Y1Q1\Research_Topics_DM\Group_Project\EMM-Friends-Project\.venv\lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [22]:
from tqdm import tqdm
tqdm.pandas()

# calculate percentage of humorous lines per episode
def calculate_humor(script, model):

    global df_dialogue_humor

    humor_count = 0
    lines = script.split('\n\n')
    dialogue_lines = [line for line in lines if ': ' in line and not line.startswith('[')]  # filter to only dialogue lines
    total_lines = 0
    
    for line in dialogue_lines:
        total_lines += 1
        dialogue = line.split(': ', 1)[1]  # get the part after the first ': '
        results = model(dialogue)
        predicted_label = max(results[0], key=lambda x: x['score'])['label']

        # add dialogue line and its humor score to df_dialogue_humor
        df_dialogue_humor = pd.concat([df_dialogue_humor, pd.DataFrame([{'dialogue_line': dialogue, 'humor': 1 if predicted_label == 'HUMOR' else 0}])], ignore_index=True)

        if predicted_label == 'HUMOR':
            humor_count += 1

    if total_lines > 0:
        return humor_count / total_lines
    return 0.0
    

# calculate humor column
df_structured['humor'] = df_structured['episode_script_clean'].progress_apply(calculate_humor, model=model_humor)

100%|██████████| 236/236 [26:42<00:00,  6.79s/it]


In [23]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,...,emotion_fear,emotion_surprise,emotion_disgust,emotion_neutral,sentiment_neg,sentiment_neu,sentiment_pos,sentiment_compound,sentiment_trajectory_slope,humor
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,...,0.046573,0.151716,0.058355,0.480126,0.069320,0.773893,0.156783,0.103839,-0.105783,0.666667
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,...,0.026094,0.170459,0.056568,0.533810,0.055897,0.786313,0.157794,0.121136,-0.058521,0.687243
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,...,0.031391,0.125838,0.070791,0.501233,0.041155,0.775074,0.183779,0.179116,-0.076444,0.674419
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,...,0.025490,0.189200,0.046026,0.503466,0.056079,0.785571,0.158349,0.111874,0.059526,0.662698
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,...,0.019409,0.176742,0.074673,0.510663,0.043924,0.796639,0.159445,0.145747,0.088462,0.731092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070,0.106583,0.153605,0.122257,...,0.021107,0.173413,0.042203,0.460592,0.070734,0.752928,0.176342,0.160163,0.039302,0.620690
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141,0.092437,0.138655,0.214286,...,0.043780,0.158035,0.033455,0.496071,0.053908,0.773298,0.172803,0.169121,0.112704,0.680672
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001,0.206107,0.167939,0.175573,...,0.017390,0.162394,0.049853,0.489490,0.046672,0.798347,0.154977,0.142221,0.020706,0.698473
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879,0.146667,0.110000,0.183333,...,0.019799,0.192423,0.034782,0.459105,0.035767,0.743953,0.220270,0.185630,0.095480,0.590000


### 4. Character Interaction

- pair_dialogues_{pair} – number of back-and-forth exchanges between character pairs (e.g., Joey–Chandler, Ross–Rachel)

#### 4.1. pair_dialogues_{pair}

In [24]:
# calculate number of back-and-forth exchanges between character pairs (e.g., Joey–Chandler, Ross–Rachel)
def count_exchanges(script, char1_list, char2_list):
    exchanges = 0
    lines = script.split('\n\n')
    dialogue_lines = [line for line in lines if ': ' in line and not line.startswith('[')]  # filter to only dialogue lines
    
    previous_speaker = None
    for line in dialogue_lines:
        speaker = line.split(': ', 1)[0]  # get the part before the first ': '
        if speaker in char1_list or speaker in char2_list:
            if previous_speaker and previous_speaker != speaker:
                if (previous_speaker in char1_list and speaker in char2_list) or (previous_speaker in char2_list and speaker in char1_list):
                    exchanges += 1
            previous_speaker = speaker
        else:
            previous_speaker = None  # reset if the speaker is not one of the two characters

    return exchanges

test_duo = [("JOEY", "CHANDLER"), ("RACHEL", "MONICA"), ("ROSS", "RACHEL"), ("MONICA", "CHANDLER"), ("PHOEBE", "JOEY")]

for char1, char2 in test_duo:
    column_name = f'exchanges_{char1}_{char2}'
    df_structured[column_name] = df_structured['episode_script'].apply(count_exchanges, char1_list=[char1], char2_list=[char2])

In [25]:
df_structured

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,...,sentiment_neu,sentiment_pos,sentiment_compound,sentiment_trajectory_slope,humor,exchanges_JOEY_CHANDLER,exchanges_RACHEL_MONICA,exchanges_ROSS_RACHEL,exchanges_MONICA_CHANDLER,exchanges_PHOEBE_JOEY
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,...,0.773893,0.156783,0.103839,-0.105783,0.666667,26,35,29,11,4
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,...,0.786313,0.157794,0.121136,-0.058521,0.687243,1,8,22,10,3
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,...,0.775074,0.183779,0.179116,-0.076444,0.674419,19,14,8,9,6
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,...,0.785571,0.158349,0.111874,0.059526,0.662698,16,31,5,2,1
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,...,0.796639,0.159445,0.145747,0.088462,0.731092,5,3,47,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,10,14,The One with Princess Consuela,[SCENE NOTE: Scene: Chandler and Monica's apar...,[SCENE NOTE: Scene: Chandler and Monica's apar...,8.934169,0.368070,0.106583,0.153605,0.122257,...,0.752928,0.176342,0.160163,0.039302,0.620690,18,4,37,15,4
232,10,15,The One Where Estelle Dies,"[SCENE NOTE: Flashback scene from last week, M...","[SCENE NOTE: Flashback scene from last week, M...",11.285714,0.377141,0.092437,0.138655,0.214286,...,0.773298,0.172803,0.169121,0.112704,0.680672,0,0,47,22,34
233,10,16,The One with Rachel's Going Away Party,[SCENE NOTE: Scene: Joey's place. Rachel and J...,[SCENE NOTE: Scene: Joey's place. Rachel and J...,10.660305,0.372001,0.206107,0.167939,0.175573,...,0.798347,0.154977,0.142221,0.020706,0.698473,17,21,21,31,9
234,10,17,The Last One: Part 1,[SCENE NOTE: Scene: Monica and Chandler's apar...,[SCENE NOTE: Scene: Monica and Chandler's apar...,8.786667,0.369879,0.146667,0.110000,0.183333,...,0.743953,0.220270,0.185630,0.095480,0.590000,3,6,29,50,27


### 5. Location Proportions

- {place}_proportion - % of where the episode revolved around

#### 5.1. {place}_proportion

In [26]:
df_rating_locations = df_rating[['season', 'episode', 'Central Perk', 'City spaces', 'Hallways',
                                 'Hospitals', 'Jobs', 'Main apartments', 'Other lodging',
                                 'Social Life', 'Transport', 'stars']]

In [27]:
# merge df_structured with df_rating_locations on season and episode
df_final = df_structured.merge(df_rating_locations, on=['season', 'episode'], how='left')

### Exporting the dataframes

In [28]:
# export df_final in a csv file in feature_engineered_data folder
df_final.to_csv('feature_engineered_data/df_final.csv', index=False)

In [29]:
df_dialogue_emotion.to_csv('dialogue_lines_results_data/df_dialogue_emotion.csv', index=False)
df_dialogue_sentiment.to_csv('dialogue_lines_results_data/df_dialogue_sentiment.csv', index=False)
df_dialogue_humor.to_csv('dialogue_lines_results_data/df_dialogue_humor.csv', index=False)